In [2]:
import math
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error
import torch
import torch.nn as nn
import torch.optim as optim

In [ ]:

# ----------------- Bước 1: Tiền xử lý dữ liệu -----------------
# Đọc dữ liệu từ file CSV
data = pd.read_csv('data.csv', parse_dates=['date'], index_col='date')
data = data.sort_index()

# Sử dụng MinMaxScaler để chuẩn hóa cột 'value'
scaler = MinMaxScaler(feature_range=(0, 1))
data['value_scaled'] = scaler.fit_transform(data[['value']])

# Hàm tạo đặc trưng dạng lag: ví dụ dùng 3 bước thời gian trước để dự báo giá trị tiếp theo
def create_lag_features(df, lags=3):
    df_features = pd.DataFrame()
    for lag in range(1, lags+1):
        df_features[f'lag_{lag}'] = df['value_scaled'].shift(lag)
    df_features['target'] = df['value_scaled']
    return df_features.dropna()

time_steps = 3
df_features = create_lag_features(data, time_steps)

# Tách đặc trưng (X) và nhãn (y)
X = df_features.drop('target', axis=1).values   # shape: (num_samples, time_steps)
y = df_features['target'].values                # shape: (num_samples,)

# Chia dữ liệu thành tập huấn luyện và tập kiểm tra theo thứ tự thời gian (80% - 20%)
split_idx = int(len(X) * 0.8)
X_train, X_test = X[:split_idx], X[split_idx:]
y_train, y_test = y[:split_idx], y[split_idx:]

# Chuyển đổi dữ liệu thành tensor và thêm dimension cho feature (input_size=1)
X_train_tensor = torch.tensor(X_train, dtype=torch.float32).unsqueeze(-1)  # shape: [batch, time_steps, 1]
X_test_tensor  = torch.tensor(X_test, dtype=torch.float32).unsqueeze(-1)
y_train_tensor = torch.tensor(y_train, dtype=torch.float32).unsqueeze(-1)
y_test_tensor  = torch.tensor(y_test, dtype=torch.float32).unsqueeze(-1)

# ----------------- Bước 2: Xây dựng mô hình Transformer -----------------
# Positional Encoding
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=100):
        super(PositionalEncoding, self).__init__()
        pe = torch.zeros(max_len, d_model)  # [max_len, d_model]
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)  # [max_len, 1]
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        pe = pe.unsqueeze(0)  # [1, max_len, d_model]
        self.register_buffer('pe', pe)
    
    def forward(self, x):
        # x: [batch_size, seq_len, d_model]
        x = x + self.pe[:, :x.size(1)]
        return x

# Transformer-based model cho dự báo chuỗi thời gian
class TransformerTimeSeries(nn.Module):
    def __init__(self, input_size, model_dim, num_heads, num_layers, dropout, output_size):
        super(TransformerTimeSeries, self).__init__()
        self.model_dim = model_dim
        # Chiếu đầu vào: từ input_size (ở đây =1) lên model_dim
        self.input_proj = nn.Linear(input_size, model_dim)
        # Positional encoding
        self.pos_encoder = PositionalEncoding(model_dim)
        # Transformer encoder
        encoder_layer = nn.TransformerEncoderLayer(d_model=model_dim, nhead=num_heads, dropout=dropout)
        self.transformer_encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        # Lớp fully connected để dự báo
        self.fc = nn.Linear(model_dim, output_size)
    
    def forward(self, x):
        # x: [batch_size, seq_len, input_size]
        x = self.input_proj(x)          # [batch_size, seq_len, model_dim]
        x = self.pos_encoder(x)         # [batch_size, seq_len, model_dim]
        # Transformer của PyTorch cần đầu vào dạng [seq_len, batch_size, model_dim]
        x = x.transpose(0, 1)           # [seq_len, batch_size, model_dim]
        x = self.transformer_encoder(x) # [seq_len, batch_size, model_dim]
        # Lấy output tại bước thời gian cuối cùng
        x = x[-1, :, :]                 # [batch_size, model_dim]
        out = self.fc(x)                # [batch_size, output_size]
        return out

# Khởi tạo mô hình với các tham số đã chọn
input_size = 1      # mỗi bước thời gian chỉ có 1 đặc trưng (value_scaled)
model_dim  = 32     # chiều của embedding
num_heads  = 4
num_layers = 2
dropout    = 0.1
output_size = 1

model = TransformerTimeSeries(input_size, model_dim, num_heads, num_layers, dropout, output_size)
print(model)

# ----------------- Bước 3: Huấn luyện mô hình -----------------
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.01)
num_epochs = 200

for epoch in range(num_epochs):
    model.train()
    optimizer.zero_grad()
    outputs = model(X_train_tensor)  # [batch_size, output_size]
    loss = criterion(outputs, y_train_tensor)
    loss.backward()
    optimizer.step()
    
    if (epoch+1) % 20 == 0:
        print(f"Epoch [{epoch+1}/{num_epochs}], Loss: {loss.item():.4f}")

# ----------------- Bước 4: Dự báo và đánh giá -----------------
model.eval()
with torch.no_grad():
    y_pred_tensor = model(X_test_tensor)
    
y_pred = y_pred_tensor.numpy()
mse = mean_squared_error(y_test, y_pred)
print(f"MSE trên tập kiểm tra: {mse:.4f}")

# Chuyển đổi dự báo về giá trị ban đầu
y_test_inv = scaler.inverse_transform(y_test_tensor.numpy())
y_pred_inv = scaler.inverse_transform(y_pred)

# Trực quan hóa kết quả dự báo
plt.figure(figsize=(8, 5))
plt.plot(y_test_inv, label='Giá trị thực', marker='o')
plt.plot(y_pred_inv, label='Dự báo', marker='x', linestyle='--')
plt.title('Dự báo chuỗi thời gian với Transformer')
plt.xlabel('Mẫu')
plt.ylabel('Giá trị')
plt.legend()
plt.show()


5. Giải thích chi tiết:
Tiền xử lý dữ liệu:

Dữ liệu được đọc từ data.csv và sắp xếp theo ngày.

Sử dụng MinMaxScaler để chuẩn hóa giá trị về khoảng [0, 1].

Hàm create_lag_features tạo ra các đặc trưng dạng lag (ở đây dùng 3 bước thời gian trước) để dự báo giá trị hiện tại.

Xây dựng mô hình Transformer:

Lớp PositionalEncoding tạo ra vector mã hóa vị trí để bổ sung thông tin thứ tự cho các bước thời gian.

Mô hình TransformerTimeSeries gồm:

input_proj: chiếu đầu vào từ 1 lên model_dim.

pos_encoder: thêm positional encoding.

transformer_encoder: sử dụng các lớp TransformerEncoderLayer để xử lý chuỗi.

fc: lớp fully connected dự báo giá trị tại bước thời gian cuối.

Đầu vào được chuyển từ định dạng [batch, seq_len, input_size] sang [seq_len, batch, model_dim] vì yêu cầu của Transformer của PyTorch.

Huấn luyện và dự báo:

Sử dụng hàm mất mát MSE và optimizer Adam để huấn luyện mô hình trong 200 epochs.

Sau huấn luyện, dự báo trên tập kiểm tra và đánh giá hiệu năng bằng MSE.

Kết quả dự báo được chuyển về giá trị ban đầu (nghịch đảo quá trình chuẩn hóa) và trực quan hóa qua biểu đồ.